In [1]:
!pip install ultralytics

  Using cached ultralytics-8.4.28-py3-none-any.whl.metadata (39 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.9 MB/s eta 0:00:00a 0:00:01


In [6]:
import os
import yaml
import torch
import torch.nn as nn
from ultralytics import YOLO

# ==========================================
# 1. ENVIRONMENT SAFETY & CLOUD BACKUP
# ==========================================
os.environ['KAGGLE_CONFIG_DIR'] = '/kaggle/working'
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_038b6b2d933a070c9902a63f8ac00cd1'

# VRAM Defragmentation
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True,garbage_collection_threshold:0.8'
torch.backends.cudnn.benchmark = False
torch.cuda.empty_cache()

def backup_to_kaggle(trainer):
    torch.cuda.empty_cache() 
    if trainer.epoch > 0 and trainer.epoch % 5 == 0:
        weights_path = f"{trainer.save_dir}/weights"
        if os.path.exists(weights_path):
            os.system(f'kaggle datasets version -p {weights_path} -m "Auto-backup Epoch {trainer.epoch}"')

# ==========================================
# 2. DATASET PATH RESOLUTION
# ==========================================
print("📂 Locating Dataset...")
dataset_path = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'images' in dirs:
        dataset_path = root
        break

if dataset_path:
    print(f"✅ Found dataset at: {dataset_path}")
    data_config = {
        'train': os.path.join(dataset_path, 'images/train'),
        'val': os.path.join(dataset_path, 'images/val'),
        'nc': 9,
        'names': ['car', 'truck', 'bus', 'motorcycle', 'bicycle', 'pedestrian', 'auto_rickshaw', 'rickshaw', 'van'] 
    }
    with open('/kaggle/working/fixed_data.yaml', 'w') as f:
        yaml.dump(data_config, f)
else:
    raise Exception("❌ Dataset not found in /kaggle/input!")

# ==========================================
# 3. DCFM_v2 DEFINITION (Safe Memory Build)
# ==========================================
class DCFM_v2(nn.Module):
    def __init__(self, *args):
        super().__init__()
        self.c2 = args[0] if (args and isinstance(args[0], int)) else 312
        self.inner = max(1, self.c2 // 4)
        self.convs = None
        self.fusion = nn.Conv2d(self.inner * 4, self.c2, 1)
        
    def _build_layers(self, c1, device):
        self.conv1 = nn.Conv2d(c1, self.inner, 3, padding=1, dilation=1).to(device)
        self.conv2 = nn.Conv2d(c1, self.inner, 3, padding=2, dilation=2).to(device)
        self.conv3 = nn.Conv2d(c1, self.inner, 3, padding=5, dilation=5).to(device)
        self.conv4 = nn.Conv2d(c1, self.inner, 1).to(device)
        self.fusion = self.fusion.to(device)
        self.convs = True

    def forward(self, x):
        if self.convs is None: self._build_layers(x.shape[1], x.device)
        out = torch.cat([self.conv1(x), self.conv2(x), self.conv3(x), self.conv4(x)], 1)
        return self.fusion(out)

import ultralytics.nn.tasks
import ultralytics.nn.modules
ultralytics.nn.tasks.DCFM_v2 = DCFM_v2
ultralytics.nn.modules.DCFM_v2 = DCFM_v2
globals()['DCFM_v2'] = DCFM_v2 

# ==========================================
# 4. ARCHITECTURE YAML GENERATION
# ==========================================
arch_yaml = """
nc: 9
scales:
  m: [0.50, 0.60, 512] 
backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 1, C3k2, [256, 1, True, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 1, C3k2, [512, 1, True, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 1, C3k2, [512, 1, True]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 1, C3k2, [512, 1, True]]
  - [-1, 1, SPPF, [512, 5]]
  - [-1, 1, Conv, [512, 1, 1]] 
head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 1, C3k2, [512, 1, True]] 
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 1, C3k2, [256, 1, True]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 1, C3k2, [512, 1, True]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 1, C3k2, [512, 1, True]]
  - [-1, 1, DCFM_v2, [312]] 
  - [[16, 19, 23], 1, Detect, [nc]] 
"""
with open('/kaggle/working/yolov11m-dcfm.yaml', 'w') as f: f.write(arch_yaml)

# ==========================================
# 5. TRAINING LAUNCH (SINGLE GPU)
# ==========================================
print("🚀 Launching SINGLE GPU training to clear DDP overhead...")
model = YOLO('/kaggle/working/yolov11m-dcfm.yaml').load('yolo11m.pt')

model.add_callback("on_train_epoch_end", backup_to_kaggle)

model.train(
    data='/kaggle/working/fixed_data.yaml',
    epochs=50, 
    imgsz=416,       
    batch=8,         
    nbs=64,          
    device=0,        # <--- CRITICAL CHANGE: Uses only ONE GPU, completely disabling DDP
    project='/kaggle/working/thesis_results', 
    name='v5_2_final',
    save=True,
    amp=True,
    cache=False,
    workers=4        
)

📂 Locating Dataset...
✅ Found dataset at: /kaggle/input/datasets/hijbullahx/masterv1
🚀 Launching SINGLE GPU training to clear DDP overhead...
Transferred 68/623 items from pretrained weights
Ultralytics 8.4.28 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/fixed_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=

OutOfMemoryError: CUDA out of memory. Tried to allocate 112.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 119.81 MiB is free. Including non-PyTorch memory, this process has 14.44 GiB memory in use. Of the allocated memory 14.20 GiB is allocated by PyTorch, and 55.01 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)